> **Historical source notebook.** This preserves the original separately written preprocessing/assisted-cropping logic. It may contain legacy local path assumptions in code cells. For portable use, prefer `src/preprocessing.py` and `src/assisted_roi_crop.py`. Historical per-image ROI coordinates were not retained.

> **Public reproducibility copy.** Notebook outputs and execution counts were cleared before public packaging to avoid local-path/account leakage. The scientific code and configuration cells are preserved. See the repository README for required data paths and reproducibility scope.

In [ ]:
# ==============================================
# Cell: Enhance (CLAHE + Gamma), Crop Fundus Only, Resize 512x512
# ==============================================
import os
import cv2
import numpy as np

# -----------------------------
# Konfigurasi
# -----------------------------
input_root  = "Normal"           # Folder dataset sumber (boleh berisi subfolder kelas)
output_root = "Normal_pre"      # Folder keluaran
target_size = (512, 512)               # (width, height)
gamma_value = 1.1                      # >0.0, contoh 0.9 (lebih gelap), 1.1 (sedikit lebih cerah)

# Ekstensi citra yang diproses
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

os.makedirs(output_root, exist_ok=True)

# -----------------------------
# Utilitas
# -----------------------------
def apply_clahe_bgr(img_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    """
    CLAHE pada channel L di ruang LAB (umum untuk fundus).
    """
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    L2 = clahe.apply(L)
    lab2 = cv2.merge([L2, A, B])
    return cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)

def apply_gamma(img_bgr, gamma=1.0):
    """
    Gamma correction (LUT) pada nilai intensitas 0..255.
    """
    if gamma <= 0:
        return img_bgr
    inv = 1.0 / gamma
    table = (np.arange(256) / 255.0) ** inv
    table = np.clip(table * 255.0, 0, 255).astype(np.uint8)
    return cv2.LUT(img_bgr, table)

def largest_circle_mask(img_bgr):
    """
    Deteksi area fundus (mendekati lingkaran terbesar) dari citra fundus.
    - Menghasilkan mask biner (fundus=255, latar=0) dan (cx, cy, r).
    Strategi:
      - HSV -> gunakan channel V (atau S) untuk threshold Otsu
      - Morfologi untuk merapikan
      - Ambil kontur terbesar, approx lingkaran via minEnclosingCircle
    """
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    V = hsv[:, :, 2]
    V_blur = cv2.GaussianBlur(V, (5, 5), 0)
    # Otsu threshold
    _, th = cv2.threshold(V_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Pastikan fundus jadi area putih: jika kebalik, invert
    # Asumsi fundus lebih terang; jika area putih terlalu kecil, invert
    if np.sum(th == 255) < 0.2 * th.size:
        th = cv2.bitwise_not(th)

    # Morfologi untuk menutup lubang & noise
    kernel = np.ones((7, 7), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=2)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)

    # Kontur terbesar
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        h, w = th.shape
        # fallback: mask bulat di tengah
        r = int(min(h, w) * 0.48)
        cy, cx = h // 2, w // 2
        mask = np.zeros_like(th)
        cv2.circle(mask, (cx, cy), r, 255, -1)
        return mask, (cx, cy, r)

    cnt = max(contours, key=cv2.contourArea)
    (cx, cy), r = cv2.minEnclosingCircle(cnt)
    cx, cy, r = int(cx), int(cy), int(r)

    mask = np.zeros_like(th)
    cv2.circle(mask, (cx, cy), r, 255, -1)
    return mask, (cx, cy, r)

def crop_to_circle(img_bgr, mask, circle):
    """
    Crop citra ke bounding box lingkaran fundus. Background di luar fundus dibuat hitam.
    """
    h, w = mask.shape
    cx, cy, r = circle
    x1 = max(cx - r, 0)
    y1 = max(cy - r, 0)
    x2 = min(cx + r, w - 1)
    y2 = min(cy + r, h - 1)

    # Terapkan mask ke citra (luar fundus -> hitam)
    img_masked = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)
    crop = img_masked[y1:y2, x1:x2]

    # Jika crop kosong, fallback ke seluruh gambar dengan mask
    if crop.size == 0:
        crop = img_masked

    return crop

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

# -----------------------------
# Pipeline per gambar
# -----------------------------
def process_image(path_in, path_out, gamma=1.1):
    img = cv2.imread(path_in)
    if img is None:
        print(f"⚠️ Gagal membaca: {path_in}")
        return False

    # 1) CLAHE (clipLimit=2.0, tileGrid=8x8)
    img = apply_clahe_bgr(img, clip_limit=2.0, tile_grid=(8, 8))

    # 2) Gamma correction
    img = apply_gamma(img, gamma=gamma)

    # 3) Deteksi fundus & masking
    mask, circle = largest_circle_mask(img)

    # 4) Crop hanya area fundus (bbox lingkaran), luar fundus hitam
    crop = crop_to_circle(img, mask, circle)

    # 5) Resize ke 512x512
    #    (biarkan aspect ratio berubah agar seragam 512x512 untuk training;
    #     jika ingin padding cara letterbox, bisa diganti sesuai kebutuhan)
    out = cv2.resize(crop, target_size, interpolation=cv2.INTER_AREA)

    # 6) Simpan
    ok = cv2.imwrite(path_out, out)
    return bool(ok)

# -----------------------------
# Traversal folder (replicate subfolder)
# -----------------------------
total, ok_cnt, fail_cnt = 0, 0, 0

for root, dirs, files in os.walk(input_root):
    # Buat path keluaran yang sesuai struktur
    rel = os.path.relpath(root, input_root)
    out_dir = os.path.join(output_root, rel) if rel != "." else output_root
    ensure_dir(out_dir)

    for fn in files:
        if not fn.lower().endswith(IMG_EXTS):
            continue
        total += 1
        in_path  = os.path.join(root, fn)
        # Simpan sebagai .png untuk konsistensi (atau pertahankan ekstensi asli jika mau)
        base, _ = os.path.splitext(fn)
        out_path = os.path.join(out_dir, base + ".png")

        if process_image(in_path, out_path, gamma=gamma_value):
            ok_cnt += 1
            print(f"✅ {in_path} -> {out_path}")
        else:
            fail_cnt += 1
            print(f"❌ Gagal memproses: {in_path}")

print(f"\nSelesai. Total: {total}, Berhasil: {ok_cnt}, Gagal: {fail_cnt}")
print("Output disimpan di:", output_root)


In [ ]:
import cv2
import os
import csv

# Konfigurasi
input_folder = "Normal_pre"         # Ganti dengan folder input
output_folder = "manual_crop/Normal"  # Folder hasil crop
csv_output = "roi_coordinatesw.csv"    # Lokasi penyimpanan koordinat
crop_size = (224, 224)                # Ukuran crop tetap

os.makedirs(output_folder, exist_ok=True)

# Buka file CSV untuk menyimpan koordinat
with open(csv_output, mode='w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Filename", "x", "y", "w", "h"])  # Header

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(input_folder, filename)
            image = cv2.imread(img_path)
            clone = image.copy()

            h_img, w_img = image.shape[:2]
            win_name = f"Geser kotak untuk crop: {filename}"
            cv2.namedWindow(win_name)
            roi_box = [w_img // 2 - crop_size[0] // 2,
                       h_img // 2 - crop_size[1] // 2,
                       crop_size[0], crop_size[1]]

            def update_box(val):
                pass  # Dummy fungsi untuk slider

            # Slider untuk memindahkan ROI
            cv2.createTrackbar('X', win_name, roi_box[0], w_img - crop_size[0], update_box)
            cv2.createTrackbar('Y', win_name, roi_box[1], h_img - crop_size[1], update_box)

            while True:
                x = cv2.getTrackbarPos('X', win_name)
                y = cv2.getTrackbarPos('Y', win_name)

                temp = clone.copy()
                cv2.rectangle(temp, (x, y), (x + crop_size[0], y + crop_size[1]), (0, 255, 0), 2)
                cv2.imshow(win_name, temp)

                key = cv2.waitKey(1) & 0xFF
                if key == ord("c"):  # Tekan 'c' untuk crop
                    crop = clone[y:y+crop_size[1], x:x+crop_size[0]]
                    save_path = os.path.join(output_folder, filename)
                    cv2.imwrite(save_path, crop)
                    writer.writerow([filename, x, y, crop_size[0], crop_size[1]])
                    break
                elif key == ord("s"):  # Tekan 's' untuk skip
                    print(f"Skipped: {filename}")
                    break
                elif key == 27:  # Esc untuk keluar
                    print("Dihentikan oleh user.")
                    exit()

            cv2.destroyAllWindows()


In [ ]:
# ==============================================
# Cell: Enhance (CLAHE + Gamma), Crop Fundus Only, Resize 512x512
# ==============================================
import os
import cv2
import numpy as np

# -----------------------------
# Konfigurasi
# -----------------------------
input_root  = "AMD_WET"           # Folder dataset sumber (boleh berisi subfolder kelas)
output_root = "AMD_WET_pre"      # Folder keluaran
target_size = (512, 512)               # (width, height)
gamma_value = 1.1                      # >0.0, contoh 0.9 (lebih gelap), 1.1 (sedikit lebih cerah)

# Ekstensi citra yang diproses
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

os.makedirs(output_root, exist_ok=True)

# -----------------------------
# Utilitas
# -----------------------------
def apply_clahe_bgr(img_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    """
    CLAHE pada channel L di ruang LAB (umum untuk fundus).
    """
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    L2 = clahe.apply(L)
    lab2 = cv2.merge([L2, A, B])
    return cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)

def apply_gamma(img_bgr, gamma=1.0):
    """
    Gamma correction (LUT) pada nilai intensitas 0..255.
    """
    if gamma <= 0:
        return img_bgr
    inv = 1.0 / gamma
    table = (np.arange(256) / 255.0) ** inv
    table = np.clip(table * 255.0, 0, 255).astype(np.uint8)
    return cv2.LUT(img_bgr, table)

def largest_circle_mask(img_bgr):
    """
    Deteksi area fundus (mendekati lingkaran terbesar) dari citra fundus.
    - Menghasilkan mask biner (fundus=255, latar=0) dan (cx, cy, r).
    Strategi:
      - HSV -> gunakan channel V (atau S) untuk threshold Otsu
      - Morfologi untuk merapikan
      - Ambil kontur terbesar, approx lingkaran via minEnclosingCircle
    """
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    V = hsv[:, :, 2]
    V_blur = cv2.GaussianBlur(V, (5, 5), 0)
    # Otsu threshold
    _, th = cv2.threshold(V_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Pastikan fundus jadi area putih: jika kebalik, invert
    # Asumsi fundus lebih terang; jika area putih terlalu kecil, invert
    if np.sum(th == 255) < 0.2 * th.size:
        th = cv2.bitwise_not(th)

    # Morfologi untuk menutup lubang & noise
    kernel = np.ones((7, 7), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=2)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)

    # Kontur terbesar
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        h, w = th.shape
        # fallback: mask bulat di tengah
        r = int(min(h, w) * 0.48)
        cy, cx = h // 2, w // 2
        mask = np.zeros_like(th)
        cv2.circle(mask, (cx, cy), r, 255, -1)
        return mask, (cx, cy, r)

    cnt = max(contours, key=cv2.contourArea)
    (cx, cy), r = cv2.minEnclosingCircle(cnt)
    cx, cy, r = int(cx), int(cy), int(r)

    mask = np.zeros_like(th)
    cv2.circle(mask, (cx, cy), r, 255, -1)
    return mask, (cx, cy, r)

def crop_to_circle(img_bgr, mask, circle):
    """
    Crop citra ke bounding box lingkaran fundus. Background di luar fundus dibuat hitam.
    """
    h, w = mask.shape
    cx, cy, r = circle
    x1 = max(cx - r, 0)
    y1 = max(cy - r, 0)
    x2 = min(cx + r, w - 1)
    y2 = min(cy + r, h - 1)

    # Terapkan mask ke citra (luar fundus -> hitam)
    img_masked = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)
    crop = img_masked[y1:y2, x1:x2]

    # Jika crop kosong, fallback ke seluruh gambar dengan mask
    if crop.size == 0:
        crop = img_masked

    return crop

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

# -----------------------------
# Pipeline per gambar
# -----------------------------
def process_image(path_in, path_out, gamma=1.1):
    img = cv2.imread(path_in)
    if img is None:
        print(f"⚠️ Gagal membaca: {path_in}")
        return False

    # 1) CLAHE (clipLimit=2.0, tileGrid=8x8)
    img = apply_clahe_bgr(img, clip_limit=2.0, tile_grid=(8, 8))

    # 2) Gamma correction
    img = apply_gamma(img, gamma=gamma)

    # 3) Deteksi fundus & masking
    mask, circle = largest_circle_mask(img)

    # 4) Crop hanya area fundus (bbox lingkaran), luar fundus hitam
    crop = crop_to_circle(img, mask, circle)

    # 5) Resize ke 512x512
    #    (biarkan aspect ratio berubah agar seragam 512x512 untuk training;
    #     jika ingin padding cara letterbox, bisa diganti sesuai kebutuhan)
    out = cv2.resize(crop, target_size, interpolation=cv2.INTER_AREA)

    # 6) Simpan
    ok = cv2.imwrite(path_out, out)
    return bool(ok)

# -----------------------------
# Traversal folder (replicate subfolder)
# -----------------------------
total, ok_cnt, fail_cnt = 0, 0, 0

for root, dirs, files in os.walk(input_root):
    # Buat path keluaran yang sesuai struktur
    rel = os.path.relpath(root, input_root)
    out_dir = os.path.join(output_root, rel) if rel != "." else output_root
    ensure_dir(out_dir)

    for fn in files:
        if not fn.lower().endswith(IMG_EXTS):
            continue
        total += 1
        in_path  = os.path.join(root, fn)
        # Simpan sebagai .png untuk konsistensi (atau pertahankan ekstensi asli jika mau)
        base, _ = os.path.splitext(fn)
        out_path = os.path.join(out_dir, base + ".png")

        if process_image(in_path, out_path, gamma=gamma_value):
            ok_cnt += 1
            print(f"✅ {in_path} -> {out_path}")
        else:
            fail_cnt += 1
            print(f"❌ Gagal memproses: {in_path}")

print(f"\nSelesai. Total: {total}, Berhasil: {ok_cnt}, Gagal: {fail_cnt}")
print("Output disimpan di:", output_root)


In [ ]:
import cv2
import os
import csv

# Konfigurasi
input_folder = "AMD_WET_pre"         # Ganti dengan folder input
output_folder = "manual_crop2/AMD_WET_pre"  # Folder hasil crop
csv_output = "roi_coordinatesw.csv"    # Lokasi penyimpanan koordinat
crop_size = (224, 224)                # Ukuran crop tetap

os.makedirs(output_folder, exist_ok=True)

# Buka file CSV untuk menyimpan koordinat
with open(csv_output, mode='w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Filename", "x", "y", "w", "h"])  # Header

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(input_folder, filename)
            image = cv2.imread(img_path)
            clone = image.copy()

            h_img, w_img = image.shape[:2]
            win_name = f"Geser kotak untuk crop: {filename}"
            cv2.namedWindow(win_name)
            roi_box = [w_img // 2 - crop_size[0] // 2,
                       h_img // 2 - crop_size[1] // 2,
                       crop_size[0], crop_size[1]]

            def update_box(val):
                pass  # Dummy fungsi untuk slider

            # Slider untuk memindahkan ROI
            cv2.createTrackbar('X', win_name, roi_box[0], w_img - crop_size[0], update_box)
            cv2.createTrackbar('Y', win_name, roi_box[1], h_img - crop_size[1], update_box)

            while True:
                x = cv2.getTrackbarPos('X', win_name)
                y = cv2.getTrackbarPos('Y', win_name)

                temp = clone.copy()
                cv2.rectangle(temp, (x, y), (x + crop_size[0], y + crop_size[1]), (0, 255, 0), 2)
                cv2.imshow(win_name, temp)

                key = cv2.waitKey(1) & 0xFF
                if key == ord("c"):  # Tekan 'c' untuk crop
                    crop = clone[y:y+crop_size[1], x:x+crop_size[0]]
                    save_path = os.path.join(output_folder, filename)
                    cv2.imwrite(save_path, crop)
                    writer.writerow([filename, x, y, crop_size[0], crop_size[1]])
                    break
                elif key == ord("s"):  # Tekan 's' untuk skip
                    print(f"Skipped: {filename}")
                    break
                elif key == 27:  # Esc untuk keluar
                    print("Dihentikan oleh user.")
                    exit()

            cv2.destroyAllWindows()
